<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>02. 🛠️ Custom SQL Database ORM from Scratch</b>
</div>


### 📌 Project Overview
In this project, we explore advanced Object-Oriented programming by writing a custom Object-Relational Mapper (ORM) from scratch.
Using Python Metaclasses to dynamically analyze class designs, and Descriptors to implement database schema validations, this project shows the backend machinery of frameworks like Django ORM and SQLAlchemy.

#### 🔑 Key Concepts Covered:
- Python Metaclasses (`__new__`) to hook into class generation and read attributes
- Data validation using Descriptors (`__get__`, `__set__`, `__set_name__`)
- Dynamic creation of SQL code based on Python attribute names
- Binding objects to an SQLite database context


In [ ]:
import sqlite3

class Field:
    """Base descriptor mapping class attributes to SQLite columns."""
    def __init__(self, db_column=None, is_primary_key=False):
        self.db_column = db_column
        self.is_primary_key = is_primary_key
        
    def __set_name__(self, owner, name):
        if not self.db_column:
            self.db_column = name
        self.name = name
        
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name, None)
        
    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value
        
class CharField(Field):
    def validate(self, value):
        if value is not None and not isinstance(value, str):
            raise TypeError(f'{self.name} must be a string. Got {type(value).__name__}')
            
class IntegerField(Field):
    def validate(self, value):
        if value is not None and not isinstance(value, int):
            raise TypeError(f'{self.name} must be an integer. Got {type(value).__name__}')

class ModelMetaclass(type):
    """Metaclass that extracts Fields and builds table metadata."""
    def __new__(mcs, name, bases, attrs):
        if name == 'Model':
            return super().__new__(mcs, name, bases, attrs)
            
        fields = {}
        pk_field = None
        
        for key, value in list(attrs.items()):
            if isinstance(value, Field):
                fields[key] = value
                if value.is_primary_key:
                    if pk_field:
                        raise ValueError('Models support only one primary key')
                    pk_field = key
                    
        attrs['_fields'] = fields
        attrs['_table_name'] = name.lower() + 's'
        attrs['_primary_key'] = pk_field
        return super().__new__(mcs, name, bases, attrs)

class Model(metaclass=ModelMetaclass):
    _db = None
    
    @classmethod
    def connect(cls, path=':memory:'):
        cls._db = sqlite3.connect(path)
        cls._db.row_factory = sqlite3.Row
        
    @classmethod
    def create_table(cls):
        col_defs = []
        for name, field in cls._fields.items():
            definition = f'{field.db_column}'
            if isinstance(field, IntegerField):
                definition += ' INTEGER'
            elif isinstance(field, CharField):
                definition += ' TEXT'
                
            if field.is_primary_key:
                definition += ' PRIMARY KEY AUTOINCREMENT'
            col_defs.append(definition)
            
        sql = f'CREATE TABLE IF NOT EXISTS {cls._table_name} ({ ", ".join(col_defs) });'
        cursor = cls._db.cursor()
        cursor.execute(sql)
        cls._db.commit()
        
    def save(self):
        cursor = self._db.cursor()
        columns = [f.db_column for name, f in self._fields.items() if not f.is_primary_key]
        values = [getattr(self, name) for name, f in self._fields.items() if not f.is_primary_key]
        
        pk_attr = self._primary_key
        pk_val = getattr(self, pk_attr) if pk_attr else None
        
        if pk_val is None:
            # INSERT
            placeholders = ', '.join(['?' for _ in columns])
            sql = f'INSERT INTO {self._table_name} ({ ", ".join(columns) }) VALUES ({placeholders})'
            cursor.execute(sql, values)
            if pk_attr:
                setattr(self, pk_attr, cursor.lastrowid)
        else:
            # UPDATE
            set_clause = ', '.join([f'{col} = ?' for col in columns])
            sql = f'UPDATE {self._table_name} SET {set_clause} WHERE {self._fields[pk_attr].db_column} = ?'
            cursor.execute(sql, values + [pk_val])
            
        self._db.commit()
        
    @classmethod
    def get_all(cls):
        cursor = cls._db.cursor()
        sql = f'SELECT * FROM {cls._table_name}'
        cursor.execute(sql)
        rows = cursor.fetchall()
        
        results = []
        for row in rows:
            instance = cls()
            for name, field in cls._fields.items():
                setattr(instance, name, row[field.db_column])
            results.append(instance)
        return results
        
    def __repr__(self):
        args = ', '.join([f'{name}={getattr(self, name)}' for name in self._fields])
        return f'<{self.__class__.__name__}({args})>'


In [ ]:
# Initialize DB Context
Model.connect(':memory:')

# 1. Define Model Schema using Metaprogramming
class Employee(Model):
    id = IntegerField(is_primary_key=True)
    name = CharField()
    department = CharField()
    salary = IntegerField()

# 2. Build table structure
Employee.create_table()
print('🛠️ SQLite table "employees" generated successfully.')

# 3. Save new objects
e1 = Employee()
e1.name = 'Faizy'
e1.department = 'Engineering'
e1.salary = 120000
e1.save()

e2 = Employee()
e2.name = 'Sarah'
e2.department = 'HR'
e2.salary = 85000
e2.save()
print('Saved employees:', e1, e2)

# 4. Validate descriptor checks (should throw TypeError)
try:
    e1.salary = 'one hundred thousand'  # Throws type error!
except TypeError as err:
    print(f'\n🚫 Type Validation Intercepted: {err}')
    
# 5. Query results
print('\nDatabase contents parsed back to Python model objects:')
for emp in Employee.get_all():
    print(f'- Employee: {emp.name} works in {emp.department} (Salary: ${emp.salary})')
